# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousseftarektalkhan2006-blip/The-ML-internsip/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in fact_content_daily_performance = one page per day (grain = report_date + client_hash_id + content_hash_id). I'm querying month=2026-03 as my mid-panel month. Dates in this partition span from 2026-03-01 to 2026-03-31


In [20]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

# The official FlyRank way: set token via DuckDB's secret mechanism
con.execute(f"""
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

query = """
SELECT
    COUNT(*) as row_count,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

result = con.execute(query).fetchdf()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date  unique_clients  unique_pages
0    9841378 2026-03-01 2026-03-31              55        331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `content_age_days`
- `days_since_last_update`

### Label
- `is_declining_label`

### Context
- `report_date`
- `client_hash_id`
- `content_hash_id`
- `gsc_data_available`

### Excluded
- `trend_direction`: Excluded because it directly describes whether performance is going up or down and would reveal the label to the model, causing target leakage.
- `trend_pct`: Excluded because it directly measures the performance change used to determine the label, also causing target leakage.
- `client_hash_id` as a predictive feature: Excluded because the identifier itself has no meaningful relationship with page performance.
- `content_hash_id` as a predictive feature: Excluded because the identifier does not describe the characteristics or performance of the page.
- Product decision fields such as `health_score`, `priority_score`, `action_type`, and `refresh_tier`: Excluded because they represent decisions or scores that could already contain the outcome we are trying to predict, creating a circular result.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# For the grain check

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
"""

result = con.execute(query).fetchdf()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


In [30]:
# This query verifies the number of rows, clients, pages, and the date range in the selected month

query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

result = con.execute(query).fetchdf()
print(result)

   row_count  unique_clients  unique_pages   min_date   max_date
0    9841378              55        331437 2026-03-01 2026-03-31


In [31]:
# Missing-value checks

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_id,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_clicks,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

result = con.execute(query).fetchdf()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  missing_report_date  missing_client_id  missing_content_id  \
0     9841378                    0                  0                   0   

   missing_impressions  missing_clicks  missing_avg_position  
0                    0               0               6230317  


In [32]:
# Check that the feature period and label period are separate.
# This helps prevent data leakage.
query = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_impressions IS NULL
    ) AS missing_gsc_impressions,

    COUNT(*) FILTER (
        WHERE gsc_clicks IS NULL
    ) AS missing_gsc_clicks,

    COUNT(*) FILTER (
        WHERE gsc_avg_position IS NULL
    ) AS missing_gsc_avg_position,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS NULL
    ) AS missing_gsc_data_available

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

result = con.execute(query).fetchdf()
print(result)

   total_rows  missing_gsc_impressions  missing_gsc_clicks  \
0     9841378                        0                   0   

   missing_gsc_avg_position  missing_gsc_data_available  
0                   6230317                           0  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

There are several limitations that should be considered when using this dataset:

- **Unbalanced history:** Not every client has data covering the same time period. Some clients have shorter histories, so comparisons across clients may not always be directly comparable.

- **Google Search Console (GSC) data availability:** Some rows may not contain usable GSC performance data. Missing or unavailable GSC data should not automatically be interpreted as zero search performance.

- **Google Analytics 4 (GA4) availability:** GA4 data is not available for every client or for every period. Values of zero may sometimes represent unavailable data rather than no user activity.

- **Time-window overlap:** When creating features and labels from daily performance data, the feature period and future label period must be kept separate. Future performance information must not be used as a feature because this would cause target leakage.

- **Decline does not prove causation:** If a page is classified as declining, the data can show that its measured performance decreased, but it cannot prove why it decreased or guarantee that refreshing the page will improve its performance.

- **Decision-support only:** The resulting score should be used to prioritize pages for human review. A high score means that a page may deserve attention first; it does not guarantee that the page will recover after being refreshed.

- **Client and content identifiers:** `client_hash_id` and `content_hash_id` are pseudonymized identifiers. They are useful for grouping and validation, but they should not be treated as meaningful predictive features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.